# CHGNet ML Structure Relaxation and CO2-to-Methanol Thermodynamics

**Purpose:** Starting from POSCAR files (CuO-C, AgO-C, WO3-C), this notebook:

1. Relaxes the three metal-oxide/graphene heterostructures with CHGNet
2. Generates and relaxes CO2RR adsorbate configurations (*COOH, *CO, *CHO, *CH2O, *CH3O)
3. Computes adsorption free energies via the Computational Hydrogen Electrode (CHE) formalism
4. Applies BEP graphene substrate corrections from DFT/PBE-D3 bilayer calculations
5. Generates free energy diagrams and an updated GPR volcano plot
6. Exports all data needed to complete the analysis notebook

**Required files in current directory:**
- `CuO-C` — POSCAR: CuO on graphene/carbon substrate
- `AgO-C` — POSCAR: AgO on graphene/carbon substrate
- `WO3-C` — POSCAR: WO3 on graphene/carbon substrate

**References:**
- CHGNet: Deng et al., *Nature Machine Intelligence* **5**, 1031 (2023)
- CHE formalism: Norskov et al., *J. Phys. Chem. B* **108**, 17886 (2004)
- ZPE+TS corrections: Peterson et al., *Energy Environ. Sci.* **3**, 1311 (2010)
- BEP corrections: this group, DFT/PBE-D3 bilayer calculations

## 0. Environment Setup

The most common failure is a broken PyTorch installation (`libtorch_global_deps.dylib` missing).
This happens when the torch wheel was downloaded incompletely or extracted incorrectly on macOS.

**Quick fix:** run Section 0a to diagnose, then Section 0b if needed, then restart kernel.

**Alternative fix in Terminal:**
```bash
conda activate ML-env
bash setup_chgnet_env.sh
```


In [ ]:
# Section 0a -- Diagnose the environment
import sys, pathlib, platform
print(f'Python: {sys.version}')
print(f'Platform: {platform.machine()} ({platform.system()})')
print()
try:
    import torch
    dylib = pathlib.Path(torch.__file__).parent / 'lib' / 'libtorch_global_deps.dylib'
    if not dylib.exists():
        print('PyTorch: BROKEN -- libtorch_global_deps.dylib missing')
        print()
        if platform.machine() == 'arm64':
            print('FIX (Apple Silicon): run in Terminal with ML-env active:')
            print('  pip uninstall torch -y')
            print('  pip install torch --index-url https://download.pytorch.org/whl/cpu')
        else:
            print('FIX (Intel Mac): run in Terminal with ML-env active:')
            print('  pip uninstall torch -y && pip install torch')
        print()
        print('Then restart kernel and re-run all cells.')
    else:
        print(f'PyTorch: OK ({torch.__version__})')
        if torch.backends.mps.is_available():
            print('  Apple MPS (GPU): available -> CHGNet will use GPU acceleration')
        else:
            print('  CPU mode (still fast on Apple M-series)')
except ImportError:
    print('PyTorch: NOT INSTALLED  -> pip install torch')
print()
for pkg in ['chgnet','pymatgen','ase','sklearn']:
    try:
        mod = __import__(pkg)
        print(f'{pkg}: OK  ({getattr(mod, "__version__", "installed")})')
    except ImportError:
        print(f'{pkg}: MISSING  -> pip install {pkg}')


In [ ]:
# Section 0b -- Fix broken PyTorch (ONLY run if Section 0a reported BROKEN)
# After this cell completes, RESTART THE KERNEL, then run from Section 1.
import subprocess, sys, platform

# print('Removing broken torch...')
# subprocess.run([sys.executable,'-m','pip','uninstall','torch','torchvision','-y'], capture_output=True)

# if platform.machine() == 'arm64':
#     cmd = [sys.executable,'-m','pip','install','torch',
#            '--index-url','https://download.pytorch.org/whl/cpu']
#     print('Apple Silicon: installing CPU wheel (MPS supported natively)')
# else:
#     cmd = [sys.executable,'-m','pip','install','torch']
#     print('Intel Mac: installing standard wheel')

# result = subprocess.run(cmd, capture_output=True, text=True)
# if result.returncode == 0:
#     print('torch reinstalled OK')
#     print('-> RESTART KERNEL NOW, then proceed from Section 1')
# else:
#     print('ERROR:'); print(result.stderr[-600:])
#     print('Fallback: run setup_chgnet_env.sh from Terminal')

# for pkg in ['chgnet','pymatgen','ase','scikit-learn']:
#     subprocess.run([sys.executable,'-m','pip','install',pkg,'-q'], capture_output=True)
#     print(f'{pkg}: installed/verified')


## 1. Imports and Configuration

In [ ]:
# Section 1 - Imports and configuration
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import AutoMinorLocator
from pathlib import Path
import json, warnings
warnings.filterwarnings('ignore')

import matplotlib as mpl
mpl.rcParams.update({
    'font.family': 'serif', 'font.size': 9, 'axes.labelsize': 10,
    'axes.linewidth': 0.9, 'xtick.direction': 'in', 'ytick.direction': 'in',
    'xtick.top': True, 'ytick.right': True,
    'legend.frameon': True, 'legend.framealpha': 0.92,
    'savefig.dpi': 300, 'pdf.fonttype': 42,
})

C_CUO='#C0392B'; C_AGO='#27AE60'
COLS = {'CuO': C_CUO, 'AgO': C_AGO}

# Active systems and structure filenames
# Files are CONTCARs from prior VASP relaxation, not generic .vasp POSCARs.
# WO3 excluded: BEP correction not yet computed (requires VASP Gr/WO3 bilayer).
POSCAR_NAMES = {
    'CuO': 'CONTCAR_CUO_graphene',
    'AgO': 'CONTCAR_AGO_graphene',
}

# BEP corrections (DFT/PBE-D3 bilayer, this group)
# d-band centre shift from graphene substrate
BEP = {'CuO': 0.515, 'AgO': 0.402}

# ZPE+TS corrections (Peterson 2010, EES, Table S2)
ZPE_TS = {
    'CO2_g':   0.30,
    'H2_g':    0.27,
    'H2O_l':   0.67,
    'COOH_s':  0.24,
    'CO_s':    0.17,
    'CHO_s':   0.22,
    'CH2O_s':  0.28,
    'CH3O_s':  0.40,
    'CH3OH_l': 0.66,
}

print('Active systems:', list(POSCAR_NAMES.keys()))
print('Filenames:     ', dict(POSCAR_NAMES))
print('BEP corrections (eV):', BEP)


## 2. Load and Validate POSCARs

In [ ]:
# Section 2 - Load and validate structure files
# POSCAR_NAMES comes from Section 1 (single source of truth - do not redefine).
from pymatgen.core import Structure
import ase.io

slabs     = {}     # pymatgen Structure objects
slabs_ase = {}     # ASE Atoms objects

def _read_structure(path: Path):
    """Read VASP-format file robustly: tries CONTCAR/POSCAR readers.
    Works for files with any name (CONTCAR_*, POSCAR_*, *.vasp, *.poscar).
    """
    # pymatgen autodetects POSCAR/CONTCAR format from content; filename
    # extension does not matter, but we pass through Structure.from_file
    # which is the most permissive entry point.
    return Structure.from_file(str(path))


def _read_ase(path: Path):
    """Read with ASE; force VASP format because the filename is non-standard."""
    return ase.io.read(str(path), format='vasp')


print(f'Loading structures from {Path.cwd()}')
print(f'Files expected: {list(POSCAR_NAMES.values())}\n')

for mat, fname in POSCAR_NAMES.items():
    p = Path(fname)
    if not p.exists():
        print(f'  WARNING: {fname} not found in {Path.cwd()}')
        print(f'    -> Set POSCAR_NAMES["{mat}"] to the correct filename')
        continue
    try:
        struct = _read_structure(p)
        atoms  = _read_ase(p)
    except Exception as exc:
        print(f'  FAILED to read {fname}: {exc}')
        continue
    slabs[mat]     = struct
    slabs_ase[mat] = atoms

    lat  = struct.lattice
    comp = struct.composition
    print(f'  {mat}: {fname}')
    print(f'    Composition  : {comp}  ({comp.reduced_formula})')
    print(f'    Sites        : {len(struct)}')
    print(f'    Cell (a,b,c) : {lat.a:.3f}  {lat.b:.3f}  {lat.c:.3f} A')
    print(f'    Angles       : {lat.alpha:.2f}  {lat.beta:.2f}  {lat.gamma:.2f} deg')
    c_sites     = [s for s in struct if s.species_string == 'C']
    metal_sites = [s for s in struct if s.species_string not in ['C','O']]
    print(f'    C atoms      : {len(c_sites)}')
    print(f'    Metal atoms  : {len(metal_sites)} ({set(s.species_string for s in metal_sites)})')
    if c_sites and metal_sites:
        z_C = np.array([s.coords[2] for s in c_sites])
        z_M = np.array([s.coords[2] for s in metal_sites])
        print(f'    z(C)         : {z_C.min():.2f} - {z_C.max():.2f} A')
        print(f'    z(Metal)     : {z_M.min():.2f} - {z_M.max():.2f} A')
    print()

# Hard sanity check: bail out if structures look identical or if loading failed.
if not slabs:
    raise FileNotFoundError(
        'No structures loaded. Check working directory and POSCAR_NAMES.')

if len(slabs) >= 2:
    keys = list(slabs)
    s0, s1 = slabs[keys[0]], slabs[keys[1]]
    if s0.composition.reduced_formula == s1.composition.reduced_formula:
        raise RuntimeError(
            f'Both structures have the same reduced composition '
            f'({s0.composition.reduced_formula}). Most likely both keys point '
            f'to the same file. Check POSCAR_NAMES.')

print(f'Loaded {len(slabs)}/{len(POSCAR_NAMES)} structures successfully.')


## 3. Structural Relaxation — Clean Slabs

Relax each heterostructure with CHGNet. The graphene layer (lowest 35% of z-range)
is held fixed to prevent substrate drift while the metal oxide surface relaxes.

Convergence criterion: F_max < 0.05 eV/A.  
Estimated time: ~2-5 min per structure on Apple Silicon M-series.

In [ ]:
# Section 3 -- Structural relaxation with CHGNet
# CHGNet: Deng et al., Nature Machine Intelligence 5, 1031 (2023)
# Universal ML interatomic potential trained on MPtrj dataset (1.5M structures)

# Version-agnostic import (CHGNet >= 0.2.0)
from chgnet.model import CHGNet
try:
    from chgnet.model.dynamics import StructOptimizer   # CHGNet >= 0.3.0
except ImportError:
    from chgnet.model.model import StructOptimizer      # CHGNet < 0.3.0
import ase.io
from ase.constraints import FixAtoms

# Load model and create optimizer
chgnet_model = CHGNet.load()
print(f'CHGNet loaded. Model params: {sum(p.numel() for p in chgnet_model.parameters()):,}')

# StructOptimizer API differs slightly between versions -- handle both
import inspect
_sig = inspect.signature(StructOptimizer.__init__)
if 'model' in _sig.parameters:
    optimizer = StructOptimizer(model=chgnet_model)   # CHGNet >= 0.3.0
else:
    optimizer = StructOptimizer()                      # CHGNet < 0.3.0

import torch
if torch.backends.mps.is_available():
    print('Apple MPS available: CHGNet will use GPU acceleration')
else:
    print('Running on CPU (fast on Apple M-series)')

# Helper: extract energy and final structure from relax result (API-safe)
def get_relax_result(result):
    """Return (final_atoms, energy, n_steps) from CHGNet relax output."""
    traj = result.get('trajectory', result.get('obs_traj', None))
    if traj is not None:
        energies = traj.energies if hasattr(traj, 'energies') else traj.get('energies', [0])
        n_steps = len(energies)
        E_final = float(energies[-1])
    else:
        E_final = float(result.get('energy', 0))
        n_steps = 0
    final = result.get('final_structure', result.get('atoms', None))
    return final, E_final, n_steps

relaxed_slabs = {}
E_slab = {}

for mat, atoms in slabs_ase.items():
    print(f'\nRelaxing {mat} ({len(atoms)} atoms)...')

    # Fix graphene layer (lowest 35% in z)
    z = atoms.get_positions()[:, 2]
    z_thresh = z.min() + (z.max() - z.min()) * 0.35
    fix_mask = z < z_thresh
    atoms.set_constraint(FixAtoms(mask=fix_mask))
    print(f'  Fixed {fix_mask.sum()} atoms (z < {z_thresh:.2f} A)')

    result = optimizer.relax(
        atoms,
        fmax=0.05,
        steps=500,
        relax_cell=False,
        verbose=False,
    )

    final_atoms, E_final, n_steps = get_relax_result(result)
    relaxed_slabs[mat] = final_atoms
    E_slab[mat] = E_final
    out = f'POSCAR_{mat}_relaxed'
    # Convert pymatgen Structure -> ASE Atoms if needed (CHGNet >= 0.3 returns Structure)
    if hasattr(final_atoms, 'sites'):
        from pymatgen.io.ase import AseAtomsAdaptor
        final_atoms = AseAtomsAdaptor.get_atoms(final_atoms)
    ase.io.write(out, final_atoms, format='vasp')
    out = f'POSCAR_{mat}_relaxed'
    ase.io.write(out, final_atoms, format='vasp')
    print(f'  Converged in {n_steps} steps,  E = {E_final:.4f} eV')
    print(f'  E/atom = {E_final/len(atoms):.4f} eV,  saved: {out}')

print('\nRelaxation complete:')
for mat, E in E_slab.items():
    print(f'  E_slab({mat}) = {E:.4f} eV')
# Integrity check: distinct slabs must produce distinct energies
_es_values = list(E_slab.values())
if len(_es_values) >= 2 and len(set(round(v, 4) for v in _es_values)) == 1:
    raise RuntimeError(
        f'E_slab has identical values for all systems: {E_slab}. '
        f'This indicates the same structure was relaxed twice. '
        f'Check POSCAR_NAMES and ensure both files exist and differ.')

print('\nE_slab populated:')
for _mat, _e in E_slab.items():
    _n = len(slabs[_mat]) if _mat in slabs else '?'
    _epa = _e / _n if isinstance(_n, int) else float('nan')
    print(f'  {_mat:8s}: {_e:.4f} eV  ({_n} atoms, {_epa:.4f} eV/atom)')


## 4. Gas-Phase Reference Molecules

Relax CO2, H2, H2O, CO as isolated molecules in large vacuum boxes.
These energies provide the absolute reference for adsorption energy calculations.

In [ ]:
# Section 4 -- Gas phase reference molecules
# Relax CO2, H2, H2O, CO as isolated molecules in vacuum boxes.

from ase import Atoms

gas_molecules = {
    'CO2': Atoms('CO2', positions=[[0,0,0],[1.16,0,0],[-1.16,0,0]],
                 cell=[15,15,15], pbc=False),
    'H2':  Atoms('H2',  positions=[[0,0,0],[0.74,0,0]],
                 cell=[12,12,12], pbc=False),
    'H2O': Atoms('H2O', positions=[[0,0,0],[0.96,0,0],[-0.24,0.93,0]],
                 cell=[12,12,12], pbc=False),
    'CO':  Atoms('CO',  positions=[[0,0,0],[1.13,0,0]],
                 cell=[12,12,12], pbc=False),
    'CH3OH': Atoms('CH4O',
                   positions=[[0,0,0],[1.43,0,0],[0,0,1.09],
                               [0,1.09,0],[-0.54,-0.54,0],[1.43,0,1.43]],
                   cell=[14,14,14], pbc=False),
}

E_gas = {}

print('Relaxing gas-phase reference molecules...')
for mol_name, mol in gas_molecules.items():
    result = optimizer.relax(mol, fmax=0.02, steps=300,
                             relax_cell=False, verbose=False)
    _, E_mol, n_steps = get_relax_result(result)
    E_gas[mol_name] = E_mol
    print(f'  E({mol_name:8s}) = {E_mol:.6f} eV  ({n_steps} steps)')

print()
print(f'  E(CO2) - E(CO) = {E_gas["CO2"] - E_gas["CO"]:.3f} eV')
print('Gas reference energies ready.')

## 5. Generate Adsorbate Configurations

Place CO2RR intermediates (*COOH, *CO, *CHO, *CH2O, *CH3O) above the topmost
metal atom of each relaxed slab. Initial geometries use standard bond lengths.

In [ ]:
# Section 5 — Adsorbate configuration generation
# Place CO2RR intermediates on the metal oxide surface.
# CRITICAL: hydrogens must point AWAY from the surface (upward), and the
# binding heavy atom (C or O) goes lowest. Wrong orientation drives the
# molecule to migrate or settle in unphysical configurations during
# relaxation, producing artificially endergonic adsorption energies
# (the *CH3O bug fixed in v6: dG_ads went from +2.40 to ~+0.05 eV).

from ase import Atoms
import ase.io
import copy as cp
from pymatgen.core import Structure

# Standard bond lengths (A) and angle helpers
D_CO_DOUBLE = 1.20    # C=O carbonyl/aldehyde
D_CO_SINGLE = 1.43    # C-O single bond (alcohol/methoxy)
D_CH        = 1.09    # methyl/methylene C-H
D_CH_FORMYL = 1.10    # aldehyde C-H
D_OH        = 0.97    # O-H
SIN_TET     = 0.943   # sin(70.5 deg) -- tetrahedral H out-of-axis
COS_TET     = 0.333   # cos(70.5 deg)
SIN_120     = 0.866
COS_120     = 0.5


def make_COOH(h=2.0):
    """*COOH: C binds to surface; one C=O and one C-OH point upward.
    Atom order: C, =O, -O, H."""
    C  = [0.0, 0.0, h]
    O1 = [-COS_120*D_CO_DOUBLE, 0.0,  h + SIN_120*D_CO_DOUBLE]
    O2 = [ COS_120*D_CO_SINGLE, 0.0,  h + SIN_120*D_CO_SINGLE]
    H  = [O2[0] + COS_120*D_OH, 0.0,  O2[2] + SIN_120*D_OH]
    return Atoms(symbols=['C','O','O','H'], positions=[C, O1, O2, H])


def make_CO(h=1.8):
    """*CO: C binds to surface, O above (linear)."""
    return Atoms(symbols=['C','O'],
                 positions=[[0,0,h], [0,0,h + D_CO_DOUBLE]])


def make_CHO(h=1.9):
    """*CHO (formyl): C binds to surface; =O up-back, H up-forward."""
    C = [0.0, 0.0, h]
    O = [-COS_120*D_CO_DOUBLE, 0.0, h + SIN_120*D_CO_DOUBLE]
    H = [ COS_120*D_CH_FORMYL, 0.0, h + SIN_120*D_CH_FORMYL]
    return Atoms(symbols=['C','O','H'], positions=[C, O, H])


def make_CH2O(h=1.9):
    """*CH2O (formaldehyde-like): C binds to surface, =O straight up,
    two methylene H atoms tilted up at +/- y (sp2 ~ 120 deg)."""
    C  = [0.0, 0.0, h]
    O  = [0.0, 0.0, h + D_CO_DOUBLE]
    H1 = [0.0,  SIN_120*D_CH, h + COS_120*D_CH]
    H2 = [0.0, -SIN_120*D_CH, h + COS_120*D_CH]
    return Atoms(symbols=['C','O','H','H'], positions=[C, O, H1, H2])


def make_CH3O(h=1.9):
    """*CH3O (methoxy): O binds to surface, C above, three methyl H atoms
    tetrahedrally arranged ABOVE C, all pointing away from surface.

    THIS IS THE FIX FOR THE +2.4 eV ANOMALY in v5.
    The previous geometry placed one H at the same z as C, which let the
    methyl group rotate into the surface during CHGNet relaxation.
    """
    O  = [0.0, 0.0, h]
    C  = [0.0, 0.0, h + D_CO_SINGLE]
    z_H = h + D_CO_SINGLE + COS_TET*D_CH
    r_H = SIN_TET*D_CH
    H1 = [ r_H,           0.0,           z_H]
    H2 = [-r_H*COS_120,   r_H*SIN_120,   z_H]
    H3 = [-r_H*COS_120,  -r_H*SIN_120,   z_H]
    return Atoms(symbols=['O','C','H','H','H'], positions=[O, C, H1, H2, H3])


adsorbate_factories = {
    'COOH': make_COOH,
    'CO':   make_CO,
    'CHO':  make_CHO,
    'CH2O': make_CH2O,
    'CH3O': make_CH3O,
}


# ── Geometry self-test (catches regressions before slab placement) ──────────
print("Adsorbate geometry self-test (binding atom at z=0):")
for name, factory in adsorbate_factories.items():
    a    = factory(h=0.0)
    syms = a.get_chemical_symbols()
    pos  = a.get_positions()
    z_bind = pos[0, 2]
    z_min  = pos[:, 2].min()
    if z_min < z_bind - 0.01:
        print(f"  {name:5s}: FAIL -- atom at z={z_min:.2f} below binding atom")
    else:
        print(f"  {name:5s}: OK  ({len(syms)} atoms, "
              f"z range {pos[:,2].min():+.3f} to {pos[:,2].max():+.3f}, "
              f"order {syms})")
print()


def find_active_site(atoms, mat):
    """Return atom index of the topmost metal atom for adsorbate placement."""
    metal_symbols = {'CuO': 'Cu', 'AgO': 'Ag', 'WO3': 'W'}
    target = metal_symbols.get(mat, None)
    if target is None:
        print(f'WARNING: unknown metal for {mat}, using highest-z atom')
        return atoms.get_positions()[:, 2].argmax()
    indices = [i for i, s in enumerate(atoms.get_chemical_symbols())
               if s == target]
    if not indices:
        print(f'WARNING: no {target} atoms found, using highest-z atom')
        return atoms.get_positions()[:, 2].argmax()
    z = atoms.get_positions()[indices, 2]
    return indices[np.argmax(z)]


def build_slab_ads(slab_atoms, adsorbate, active_idx):
    """Place adsorbate so its binding atom (index 0) sits above the active site."""
    slab     = cp.deepcopy(slab_atoms)
    site_pos = slab.get_positions()[active_idx]
    ads_pos  = adsorbate.get_positions().copy()
    ads_pos[:, 0] += site_pos[0] - ads_pos[0, 0]
    ads_pos[:, 1] += site_pos[1] - ads_pos[0, 1]
    ads_pos[:, 2] += site_pos[2]
    combined = slab + Atoms(
        symbols   = adsorbate.get_chemical_symbols(),
        positions = ads_pos,
        cell      = slab.get_cell(),
        pbc       = slab.get_pbc(),
    )
    return combined


# Generate all slab+adsorbate configurations
slab_ads_configs = {}
active_sites     = {}

# Ensure relaxed_slabs contains ASE Atoms (CHGNet >= 0.3 returns pymatgen Structure)
from pymatgen.io.ase import AseAtomsAdaptor
for mat in list(relaxed_slabs.keys()):
    if hasattr(relaxed_slabs[mat], 'sites'):
        relaxed_slabs[mat] = AseAtomsAdaptor.get_atoms(relaxed_slabs[mat])

for mat, slab in relaxed_slabs.items():
    active_idx        = find_active_site(slab, mat)
    active_sites[mat] = active_idx
    site_pos          = slab.get_positions()[active_idx]
    print(f"{mat}: active site = atom {active_idx} "
          f"({slab.get_chemical_symbols()[active_idx]}) at z={site_pos[2]:.2f} A")

    slab_ads_configs[mat] = {}
    for ads_name, factory in adsorbate_factories.items():
        ads     = factory(h=2.0)
        config  = build_slab_ads(slab, ads, active_idx)
        slab_ads_configs[mat][ads_name] = config

        # Sanity: verify the binding atom sits above the metal, and that
        # all hydrogens are above the binding atom.
        n_ads        = len(ads)
        ads_pos      = config.get_positions()[-n_ads:]
        ads_syms     = config.get_chemical_symbols()[-n_ads:]
        z_bind       = ads_pos[0, 2]
        z_above_bind = ads_pos[1:, 2]
        h_indices    = [i for i, s in enumerate(ads_syms) if s == 'H']
        if z_bind < site_pos[2] - 0.5:
            print(f"  WARNING {mat}+*{ads_name}: binding atom below metal "
                  f"(z={z_bind:.2f}, z_metal={site_pos[2]:.2f})")
        if h_indices and ads_pos[h_indices, 2].min() < z_bind - 0.1:
            print(f"  WARNING {mat}+*{ads_name}: H atom below binding atom "
                  f"(z_H_min={ads_pos[h_indices, 2].min():.2f}, "
                  f"z_bind={z_bind:.2f})")
        print(f"  Built {mat}+*{ads_name}: {len(config)} atoms, "
              f"ads z-range {ads_pos[:,2].min():.2f} to {ads_pos[:,2].max():.2f} A")

print()
print(f"Total configurations: {sum(len(v) for v in slab_ads_configs.values())}")
print("Ready for relaxation in Section 6.")


## 6. Relax Slab + Adsorbate Configurations

Most computationally expensive section. Results are cached to
`chgnet_results_cache.json` so interrupted runs can be resumed.

Estimated time: 1-10 min per configuration (3 materials x 5 intermediates = 15 runs).
Total: ~15-90 min depending on system size and hardware.

In [ ]:
# Section 6 -- Relax slab+adsorbate configurations
# Results cached to chgnet_results_cache.json (safe to interrupt and resume).

from ase.constraints import FixAtoms
import ase.io, json as _json
from pathlib import Path

CACHE = 'chgnet_results_cache.json'
cache = _json.loads(Path(CACHE).read_text()) if Path(CACHE).exists() else {}
print(f'Cache: {len(cache)} entries already computed')

E_ads_raw = {}
converged  = {}

for mat in slab_ads_configs:
    E_ads_raw[mat] = {}
    converged[mat]  = {}

    for ads_name, config in slab_ads_configs[mat].items():
        key = f'{mat}_{ads_name}'

        if key in cache:
            E_ads_raw[mat][ads_name] = cache[key]['energy']
            converged[mat][ads_name] = cache[key]['converged']
            print(f'  {key}: cached  E={E_ads_raw[mat][ads_name]:.4f} eV')
            continue

        print(f'  Relaxing {key} ({len(config)} atoms)...', end='', flush=True)

        # Fix graphene layer
        z = config.get_positions()[:, 2]
        z_thresh = z.min() + (z.max()-z.min())*0.35
        config.set_constraint(FixAtoms(mask=z < z_thresh))

        try:
            result = optimizer.relax(config, fmax=0.05, steps=400,
                                     relax_cell=False, verbose=False)
            final_atoms, E_final, n_steps = get_relax_result(result)
            # ase.io.write(f'POSCAR_{mat}_{ads_name}_relaxed', final_atoms, format='vasp')
            if hasattr(final_atoms, 'sites'):
                from pymatgen.io.ase import AseAtomsAdaptor
                final_atoms = AseAtomsAdaptor.get_atoms(final_atoms)
            ase.io.write(f'POSCAR_{mat}_{ads_name}_relaxed', final_atoms, format='vasp')
            print(f' {n_steps} steps  E={E_final:.4f} eV')
            conv = True
        except Exception as e:
            print(f' FAILED: {e}')
            E_final = float('nan')
            conv = False

        E_ads_raw[mat][ads_name] = E_final
        converged[mat][ads_name] = conv
        cache[key] = {'energy': E_final, 'converged': conv}
        Path(CACHE).write_text(_json.dumps(cache, indent=2))

print()
print('Raw energies (eV):')
for mat in E_ads_raw:
    for ads, E in E_ads_raw[mat].items():
        status = 'OK' if converged[mat][ads] else 'FAILED'
        print(f'  E({mat}+*{ads}) = {E:.4f}  [{status}]')

## 7. Compute Adsorption Free Energies

Apply the CHE formalism with ZPE+TS corrections from Peterson 2010.
All energies referenced to CO2(g) + H2(g) at U=0, pH=0.

In [ ]:
# Section 7 — Adsorption free energies via CHE (corrected E_refs)
#
# BUG FIXED: CHO/CH2O/CH3O previously used E_gas['CO'] as sub-reference.
# CHGNet WGS error (CO+H2O->CO2+H2) introduces a systematic ~0.1-0.3 eV
# offset into every step referencing CO. Direct stoichiometric refs avoid this.
#
# CORRECT E_refs (direct stoichiometry from CO2/H2/H2O only):
#   *COOH : E_CO2 + 0.5*E_H2
#   *CO   : E_CO2 + E_H2 - E_H2O
#   *CHO  : E_CO2 + 1.5*E_H2 - E_H2O
#   *CH2O : E_CO2 + 2.0*E_H2 - E_H2O
#   *CH3O : E_CO2 + 2.5*E_H2 - E_H2O

# ── ZPE+TS net corrections per step (ΔG - ΔE, eV at 298 K) ─────────────────
# Source: Peterson et al. 2010, EES, Table S2
# With direct stoichiometric E_refs, DZPE must also use the same chain:
# DZPE[X] = ZPE_TS[X_s] - ZPE_TS[CO2_g] - n_H*ZPE_TS[H2_g] + n_H2O*ZPE_TS[H2O_l]
DZPE = {
    'COOH': ZPE_TS['COOH_s'] - ZPE_TS['CO2_g'] - 0.5*ZPE_TS['H2_g'],
    'CO':   ZPE_TS['CO_s']   - ZPE_TS['CO2_g'] - 1.0*ZPE_TS['H2_g'] + ZPE_TS['H2O_l'],
    'CHO':  ZPE_TS['CHO_s']  - ZPE_TS['CO2_g'] - 1.5*ZPE_TS['H2_g'] + ZPE_TS['H2O_l'],
    'CH2O': ZPE_TS['CH2O_s'] - ZPE_TS['CO2_g'] - 2.0*ZPE_TS['H2_g'] + ZPE_TS['H2O_l'],
    'CH3O': ZPE_TS['CH3O_s'] - ZPE_TS['CO2_g'] - 2.5*ZPE_TS['H2_g'] + ZPE_TS['H2O_l'],
}
print('Net ZPE+TS corrections (eV):')
for k,v in DZPE.items(): print(f'  {k:8s}: {v:+.3f}')

# ── Shorthand gas refs ────────────────────────────────────────────────────
E_CO2 = E_gas['CO2']; E_H2 = E_gas['H2']
E_H2O = E_gas['H2O']; E_CO  = E_gas['CO']

# ── WGS consistency check ────────────────────────────────────────────────
dE_wgs = E_CO2 + E_H2 - E_CO - E_H2O
print(f'\nWater-gas shift (CHGNet): {dE_wgs:+.3f} eV  (exp ~-0.41 eV, err={dE_wgs+0.41:+.3f})')
if abs(dE_wgs + 0.41) < 0.20:
    print('  WGS error < 0.20 eV: CHGNet molecular energetics are reasonable')
else:
    print('  WGS error > 0.20 eV: systematic offset in CO/H2O energies')
    print('  CHO/CH2O/CH3O steps affected — direct stoichiometric E_refs used (see above)')

# ── Adsorption energies ──────────────────────────────────────────────────
dE_ads = {}
dG_ads = {}

print(f'\nAdsorption free energies dG (eV):')
print(f'  {"System":<16} {"*COOH":>8} {"*CO":>8} {"*CHO":>8} {"*CH2O":>8} {"*CH3O":>8}')
print('  '+'-'*56)

for mat in E_slab:
    dE_ads[mat] = {}
    dG_ads[mat] = {}
    Es = E_slab[mat]
    Er = E_ads_raw[mat]

    # Direct stoichiometric E_refs — absolute CHGNet energies cancel in step dG
    dE_ads[mat]['COOH'] = Er.get('COOH', np.nan) - Es - E_CO2 - 0.5*E_H2
    dE_ads[mat]['CO']   = Er.get('CO',   np.nan) - Es - E_CO2 - 1.0*E_H2 + E_H2O
    dE_ads[mat]['CHO']  = Er.get('CHO',  np.nan) - Es - E_CO2 - 1.5*E_H2 + E_H2O
    dE_ads[mat]['CH2O'] = Er.get('CH2O', np.nan) - Es - E_CO2 - 2.0*E_H2 + E_H2O
    dE_ads[mat]['CH3O'] = Er.get('CH3O', np.nan) - Es - E_CO2 - 2.5*E_H2 + E_H2O

    for ads in ['COOH','CO','CHO','CH2O','CH3O']:
        dG_ads[mat][ads] = dE_ads[mat][ads] + DZPE[ads]

    row = '  '.join(f'{dG_ads[mat][k]:>8.3f}' for k in ['COOH','CO','CHO','CH2O','CH3O'])
    print(f'  {mat:<16} {row}')

print('\nExpected ranges (literature CO2RR on oxide surfaces, eV):')
print('  *COOH: -2.0 to +0.5   *CO: -2.5 to +0.5   *CHO: -2.5 to +0.5')
print('  *CH2O: -2.0 to +0.5   *CH3O: -2.5 to +0.5')
print('Note: BEP correction applied in Section 8. These are bare surface values.')


## 8. Apply BEP Graphene Substrate Corrections

The BEP (Bronsted-Evans-Polanyi) correction accounts for the d-band centre shift
induced by the graphene substrate:

    dG_corr = dG_bare + beta * Delta_eps_d

where Delta_eps_d is from DFT/PBE-D3 bilayer calculations (this group):
- CuO: +0.515 eV
- AgO: +0.402 eV
- WO3: None (pending VASP relaxation of Gr/WO3 bilayer)

In [ ]:
# Section 8 — BEP graphene substrate corrections
# dG_corr = dG_bare + Delta_eps_d
# Delta_eps_d: DFT/PBE-D3 bilayer (this group)
# CuO: +0.515 eV  AgO: +0.402 eV

dG_corr = {}
print('Applying BEP corrections:')
for mat in dG_ads:
    bep_val = BEP[mat]
    dG_corr[mat] = {ads: dG_ads[mat][ads] + bep_val
                   for ads in dG_ads[mat]}
    print(f'  {mat}: +{bep_val:.3f} eV applied')
    print(f'    dG(*CO) bare = {dG_ads[mat]["CO"]:.3f}  '
          f'corr = {dG_corr[mat]["CO"]:.3f} eV')

print('\nBEP-corrected dG(*CO) [volcano descriptor]:')
for mat, vals in dG_corr.items():
    print(f'  {mat}: dG(*CO) = {vals["CO"]:.3f} eV')


## 9. Free Energy Diagrams

Generate cumulative free energy profiles for CO2 -> CH3OH at three potentials:
- U = 0 V (standard conditions)
- U = -0.38 V vs RHE (CO2/CH3OH thermodynamic threshold)
- U = U_L (limiting potential, all steps downhill)

In [ ]:
# Section 9 — Free energy diagrams (CHE formalism)
# Norskov et al. J. Phys. Chem. B 108, 17886 (2004)
# G(H+ + e-) = 0.5*G(H2) - eU  at standard pH=0

def free_energy_profile(dG_dict, U_V=0.0, pH=0.0):
    # Returns cumulative free energies for CO2 -> CH3OH
    # Steps:
    # 0  CO2(g)
    # 1  *COOH        CO2 + H+ + e- -> *COOH
    # 2  *CO + H2O    *COOH + H+ + e- -> *CO + H2O
    # 3  *CHO         *CO + H+ + e- -> *CHO
    # 4  *CH2O        *CHO + H+ + e- -> *CH2O
    # 5  *CH3O        *CH2O + H+ + e- -> *CH3O
    # 6  CH3OH        *CH3O + H+ + e- -> CH3OH + *
    dG_H = U_V   # G(H+ + e-) = -eU per step (negative = driving force at cathodic U)

    G = np.zeros(7)
    # Step 1: CO2 -> *COOH
    G[1] = G[0] + dG_dict['COOH'] + dG_H
    # Step 2: *COOH -> *CO + H2O
    # ΔG_step2 = dG(*CO) - dG(*COOH) + dG(H2O formation) + dG_H
    # dG(H2O) from H2 + 1/2 O2 -> H2O = -2.46 eV at standard (pH 0)
    # In CHE: already absorbed into the CO reference (CO2+H2O<->CO path)
    # Simpler: ΔG2 = dG_CO - dG_COOH + dG_H + G_H2O_correction
    # G_H2O_corr = ZPE_TS(H2O) - ZPE_TS(H2) - 0.5*ZPE_TS(O2) = 0.67-0.27 ~ 0.40 eV
    G_H2O_corr = ZPE_TS['H2O_l'] - ZPE_TS['H2_g']
    G[2] = G[1] + (dG_dict['CO'] - dG_dict['COOH']) + G_H2O_corr + dG_H
    # Step 3: *CO -> *CHO
    G[3] = G[2] + (dG_dict['CHO'] - dG_dict['CO']) + dG_H
    # Step 4: *CHO -> *CH2O
    G[4] = G[3] + (dG_dict['CH2O'] - dG_dict['CHO']) + dG_H
    # Step 5: *CH2O -> *CH3O
    G[5] = G[4] + (dG_dict['CH3O'] - dG_dict['CH2O']) + dG_H
    # Step 6: *CH3O -> CH3OH  (ΔG6 = G(CH3OH) - dG(*CH3O))
    # G(CH3OH) relative to CO2 + 3H2: ΔG_rxn(CO2+3H2->CH3OH+H2O) = -0.13 eV (exergonic)
    # At standard: G_final = -0.13 eV (reaction Gibbs energy, NIST)
    G_rxn_total = -0.13  # eV  (CO2 + 3H2 -> CH3OH + H2O, NIST)
    G[6] = G[0] + G_rxn_total - 6*dG_H  # thermodynamic endpoint
    return G

def limiting_potential(dG_dict):
    # U_L = -max(ΔG_steps) / e   (at U=0)
    G = free_energy_profile(dG_dict, U_V=0.0)
    step_dG = np.diff(G)
    return -float(np.max(step_dG))  # V vs RHE

# Compute limiting potentials
U_L = {}
for mat, dG in dG_corr.items():
    if any(np.isnan(v) for v in dG.values()):
        U_L[mat] = np.nan
        print(f'{mat}: U_L = NaN (incomplete data)')
    else:
        U_L[mat] = limiting_potential(dG)
        print(f'{mat}: U_L = {U_L[mat]:.3f} V vs RHE')

print()
print('Literature reference: Cu(211) U_L ~ -0.52 V vs RHE (Peterson 2010)')

In [ ]:
# Section 9b — Plot free energy diagrams
from matplotlib.lines import Line2D

step_labels = ['CO$_2$(g)', '*COOH', '*CO', '*CHO', '*CH$_2$O', '*CH$_3$O', 'CH$_3$OH']
# NOTE: if matplotlib math issues, use plain text:
step_labels_plain = ['CO2(g)', '*COOH', '*CO', '*CHO', '*CH2O', '*CH3O', 'CH3OH']

n_mats = len(dG_corr)
fig, axes = plt.subplots(1, n_mats, figsize=(5*n_mats, 5.5), sharey=False)
if n_mats == 1:
    axes = [axes]
fig.subplots_adjust(wspace=0.35)

U_values = [0.0, -0.38, None]   # U=0, CO2/CH3OH threshold, U=U_L
# U_labels  = ['U = 0 V', 'U = -0.38 V (CO2/MeOH)', 'U = U_L']
U_labels = [
    r'$U = 0\ \mathrm{V}$',
    r'$U = -0.38\ \mathrm{V}\ (\mathrm{CO_2/MeOH})$',
    r'$U = U_L$'
]
U_colors  = ['#2C3E50', 'steelblue', C_CUO]
U_styles  = ['-', '--', ':']

for ax_i, (mat, dG) in enumerate(dG_corr.items()):
    ax = axes[ax_i]
    col = COLS[mat]
    x = np.arange(7)

    for u_val, u_lab, u_col, u_sty in zip(U_values, U_labels, U_colors, U_styles):
        if u_val is None:
            u_val = U_L.get(mat, np.nan)
            if np.isnan(u_val): continue

        if any(np.isnan(v) for v in dG.values()):
            ax.text(0.5, 0.5, 'Incomplete data\n(NaN in dG)',
                    ha='center', va='center', transform=ax.transAxes, fontsize=10)
            continue

        G = free_energy_profile(dG, U_V=u_val)

        # Draw as step function
        for i in range(6):
            ax.plot([x[i], x[i]+0.9], [G[i], G[i]], '-', color=u_col, lw=2.0, alpha=0.85)
            ax.plot([x[i]+0.9, x[i]+0.9], [G[i], G[i+1]], '--', color=u_col, lw=0.8, alpha=0.5)
        ax.plot([x[6], x[6]+0.9], [G[6], G[6]], '-', color=u_col, lw=2.0, alpha=0.85,
                label=f'{u_lab}: U_L={U_L.get(mat, np.nan):.2f} V' if u_val==U_L.get(mat) else u_lab)

    ax.axhline(0, color='0.7', lw=0.5, ls=':')
    ax.set_xticks(x + 0.45)
    ax.set_xticklabels(step_labels_plain, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('Free energy (eV)')
    # ax.set_title(f'{mat}/Graphene\nU_L = {U_L.get(mat, np.nan):.3f} V vs RHE')
    ax.set_title(f'{mat}/Graphene\n$U_L = {U_L.get(mat, np.nan):.3f}\\,\\mathrm{{V\\ vs\\ RHE}}$')
    ax.legend(fontsize=7.5, loc='best')
    ax.yaxis.set_minor_locator(AutoMinorLocator(4))
    ax.set_xlim(-0.3, 7.2)

fig.suptitle('CO2 to Methanol Free Energy Diagrams\nCHGNet + BEP graphene correction', fontsize=11, y=1.02)
plt.savefig('Fig_FED_CO2_methanol.pdf', bbox_inches='tight')
plt.show(); print('Saved Fig_FED_CO2_methanol.pdf')

## 10. Updated GPR Volcano Plot

Place the CHGNet-computed oxide positions on the Hammer-Norskov volcano
trained on literature DFT values for metals.

In [ ]:
# Section 10 — Update GPR volcano with computed oxide positions
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel, WhiteKernel

# Literature training data (from analysis notebook, Peterson/Feaster/Bhatt/Han)
lit_training = [
    ('Ni(111)',  -1.41, -1.00), ('Fe(110)',  -1.25, -0.95), ('Co(0001)', -1.05, -0.88),
    ('Cu(211)',  -0.74, -0.52), ('Cu(111)',  -0.45, -0.70),
    ('Au(111)',  +0.18, -0.79), ('Zn',       +0.33, -0.78), ('Ag(111)',  +0.35, -0.85),
    ('Sn',       +0.63, -0.91), ('In',       +0.70, -0.97),
    ('Pb',       +0.85, -1.05), ('Bi',       +0.92, -1.10),
]
dG_train = np.array([r[1] for r in lit_training]).reshape(-1,1)
UL_train = np.array([r[2] for r in lit_training])

kernel = (ConstantKernel(1.0,(0.1,10)) *
          Matern(length_scale=0.8, length_scale_bounds=(0.2,3.0), nu=2.5) +
          WhiteKernel(noise_level=0.005, noise_level_bounds=(1e-5,0.05)))
gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=15,
                                normalize_y=True, alpha=1e-6)
gpr.fit(dG_train, UL_train)

X_pred = np.linspace(-1.9, 1.5, 500).reshape(-1,1)
UL_pred, UL_std = gpr.predict(X_pred, return_std=True)

# Oxide volcano positions (from CHGNet + BEP correction)
oxide_positions = {}
for mat, dG in dG_corr.items():
    co_val = dG.get('CO', np.nan)
    ul_val = U_L.get(mat, np.nan)
    oxide_positions[mat] = {'dG_CO': co_val, 'UL': ul_val}

print('Oxide volcano positions (dG(*CO) descriptor, BEP-corrected):')
for mat, pos in oxide_positions.items():
    bep_status = 'CORRECTED' if BEP.get(mat) is not None else 'UNCORRECTED'
    print(f'  {mat}: dG(*CO)={pos["dG_CO"]:.3f} eV  U_L={pos["UL"]:.3f} V  [{bep_status}]')

In [ ]:
# Section 10b — Plot updated volcano
fig, ax = plt.subplots(figsize=(9, 5.5))

ax.fill_between(X_pred.ravel(), UL_pred-2*UL_std, UL_pred+2*UL_std,
                color='0.80', alpha=0.50, label='GPR +/- 2sigma')
ax.fill_between(X_pred.ravel(), UL_pred-UL_std,   UL_pred+UL_std,
                color='0.65', alpha=0.60, label='GPR +/- 1sigma')
ax.plot(X_pred, UL_pred, 'k-', lw=2.0, label='GPR mean')

# Literature training points
# for dg, ul, lbl in lit_training:
for lbl, dg, ul in lit_training:
    ax.scatter(dg, ul, color='0.40', s=45, zorder=5, edgecolors='0.2', linewidths=0.6)
    ax.text(dg+0.04, ul+0.02, lbl, fontsize=7, color='0.3', va='bottom', ha='left')

# Oxide positions from CHGNet
for mat, pos in oxide_positions.items():
    dg = pos['dG_CO']; ul = pos['UL']
    col = COLS[mat]
    if not np.isnan(dg) and not np.isnan(ul):
        ax.scatter(dg, ul, color=col, s=120, marker='D', zorder=8, edgecolors='k',
                   linewidths=0.8, label=f'{mat}/Gr (CHGNet+BEP)')
        ax.annotate(mat, (dg, ul), xytext=(dg+0.06, ul+0.04), fontsize=9,
                    color=col, fontweight='bold')
    else:
        bep_note = 'BEP pending' if BEP.get(mat) is None else 'data incomplete'
        ax.scatter([], [], color=col, s=80, marker='D',
                   label=f'{mat}/Gr (no data)')

# Reference lines
ax.axhline(-0.38, color='steelblue', ls='--', lw=1.0, alpha=0.8)
ax.axhline( 0.00, color='tomato',    ls=':',  lw=0.8, alpha=0.7)
ax.text(1.3, -0.34, 'CO2/CH3OH  -0.38 V', ha='right', fontsize=7.5, color='steelblue')
ax.text(1.3,  0.03, 'H+/H2  0 V',         ha='right', fontsize=7.5, color='tomato')

# ax.set_xlabel('dG(*CO) (eV)  [BEP-corrected, graphene substrate]')
ax.set_xlabel(r'$\Delta G_{\mathrm{*CO}}$ (eV) [BEP-corrected, graphene substrate]')
# ax.set_ylabel('Limiting potential U_L (V vs RHE)')
ax.set_ylabel(r'Limiting potential $U_L$ ($\mathrm{V\,vs\,RHE}$)')
ax.set_title('Hammer-Norskov GPR Volcano: CO2 to CH3OH\nLiterature metals (GPR) + oxide/graphene (CHGNet+BEP)')
# ax.set_xlim(-1.9, 1.5); ax.set_ylim(-1.3, 0.3)
all_dg = [r[1] for r in lit_training] + [pos['dG_CO'] for pos in oxide_positions.values() if not np.isnan(pos['dG_CO'])]
all_ul = [r[2] for r in lit_training] + [pos['UL']    for pos in oxide_positions.values() if not np.isnan(pos['UL'])]
pad = 0.25
ax.set_xlim(min(all_dg)-pad, max(all_dg)+pad)
ax.set_ylim(min(all_ul)-pad, max(all_ul)+pad)
ax.xaxis.set_minor_locator(AutoMinorLocator(4))
ax.yaxis.set_minor_locator(AutoMinorLocator(4))
ax.legend(fontsize=10, loc='lower left', ncol=2)

plt.tight_layout()
plt.savefig('Fig_Volcano_updated.pdf', bbox_inches='tight')
plt.show(); print('Saved Fig_Volcano_updated.pdf')

In [ ]:
# Section 10b — Plot updated volcano
fig, ax = plt.subplots(figsize=(9.5, 5.8))

ax.fill_between(
    X_pred.ravel(), UL_pred - 2*UL_std, UL_pred + 2*UL_std,
    color='0.80', alpha=0.50, label=r'GPR $\pm$ 2$\sigma$'
)
ax.fill_between(
    X_pred.ravel(), UL_pred - UL_std, UL_pred + UL_std,
    color='0.65', alpha=0.60, label=r'GPR $\pm$ 1$\sigma$'
)
ax.plot(X_pred, UL_pred, 'k-', lw=2.2, label='GPR mean')

# Literature training points
for lbl, dg, ul in lit_training:
    ax.scatter(
        dg, ul, color='0.40', s=55, zorder=5,
        edgecolors='0.2', linewidths=0.6
    )
    ax.text(
        dg + 0.04, ul + 0.025, lbl,
        fontsize=9, color='0.25', va='bottom', ha='left'
    )

# Oxide positions from CHGNet
for mat, pos in oxide_positions.items():
    dg = pos['dG_CO']
    ul = pos['UL']
    col = COLS[mat]

    if not np.isnan(dg) and not np.isnan(ul):
        ax.scatter(
            dg, ul, color=col, s=135, marker='D', zorder=8,
            edgecolors='k', linewidths=0.9,
            label=f'{mat}/Gr (CHGNet+BEP)'
        )
        ax.annotate(
            mat, (dg, ul),
            xytext=(dg + 0.06, ul + 0.045),
            fontsize=10, color=col, fontweight='bold'
        )
    else:
        bep_note = 'BEP pending' if BEP.get(mat) is None else 'data incomplete'
        ax.scatter(
            [], [], color=col, s=90, marker='D',
            label=f'{mat}/Gr (no data)'
        )

# Reference lines
ax.axhline(-0.38, color='steelblue', ls='--', lw=1.1, alpha=0.85)
ax.axhline(0.00,  color='tomato',    ls=':',  lw=0.9, alpha=0.75)

ax.text(
    1.3, -0.34,
    r'CO$_2$/CH$_3$OH   $-0.38$ V',
    ha='right', fontsize=10, color='steelblue'
)
ax.text(
    1.3, 0.035,
    r'H$^+$/H$_2$   $0$ V',
    ha='right', fontsize=10, color='tomato'
)

ax.set_xlabel(
    r'$\Delta G_{\mathrm{*CO}}$ (eV) [BEP-corrected, graphene substrate]',
    fontsize=14
)
ax.set_ylabel(
    r'Limiting potential $U_L$ ($\mathrm{V\,vs\,RHE}$)',
    fontsize=14
)
ax.set_title(
    r'Hammer--Norskov GPR Volcano: CO$_2$ to CH$_3$OH'
    '\n'
    r'Literature metals (GPR) + oxide/graphene (CHGNet+BEP)',
    fontsize=15
)

all_dg = [r[1] for r in lit_training] + [
    pos['dG_CO'] for pos in oxide_positions.values()
    if not np.isnan(pos['dG_CO'])
]
all_ul = [r[2] for r in lit_training] + [
    pos['UL'] for pos in oxide_positions.values()
    if not np.isnan(pos['UL'])
]

pad_x = 0.25
pad_y = 0.25
ax.set_xlim(min(all_dg) - pad_x, max(all_dg) + pad_x)

# Increase positive y-axis limit
ymin = min(all_ul) - pad_y
ymax = max(all_ul) + 0.45   # increased upper limit
ax.set_ylim(ymin, ymax+0.4)

ax.xaxis.set_minor_locator(AutoMinorLocator(4))
ax.yaxis.set_minor_locator(AutoMinorLocator(4))

# Bigger tick labels
ax.tick_params(axis='both', which='major', labelsize=12, length=6)
ax.tick_params(axis='both', which='minor', length=3)

ax.legend(fontsize=10.5, loc='lower left', ncol=2, frameon=True)

plt.tight_layout()
plt.savefig('Fig_Volcano_updated.pdf', bbox_inches='tight')
plt.show()
print('Saved Fig_Volcano_updated.pdf')

In [ ]:
# Diagnostic: print exact oxide_positions values
print("oxide_positions:")
for mat, pos in oxide_positions.items():
    print(f"  {mat}: dG_CO={pos['dG_CO']:.4f}  UL={pos['UL']:.4f}")

print()
print(f"Volcano xlim: (-1.9, 1.5)")
print(f"Volcano ylim: (-1.3, 0.3)")
for mat, pos in oxide_positions.items():
    dg = pos['dG_CO']; ul = pos['UL']
    in_x = -1.9 <= dg <= 1.5
    in_y = -1.3 <= ul <= 0.3
    print(f"  {mat}: in_x={in_x}  in_y={in_y}  nan={np.isnan(dg) or np.isnan(ul)}")

## 11. Export All Results

In [ ]:
# Section 11 — Export results in two formats:
#   1. chgnet_analysis_results.json   (existing format, for §12 paste block)
#   2. report_CHGNet.json             (merge_results.py schema)

import json
from pathlib import Path

# ── Format 1: full details (used by analysis notebook JSON loader) ──────
output = {
    'model': 'CHGNet',
    'structures': {
        mat: {'poscar_relaxed': f'POSCAR_{mat}_relaxed',
              'n_atoms': len(relaxed_slabs[mat]),
              'E_slab_eV': E_slab[mat]}
        for mat in relaxed_slabs
    },
    'gas_references': {k: float(v) for k,v in E_gas.items()},
    'adsorption_energies_raw': {
        mat: {ads: float(v) for ads,v in vals.items()}
        for mat,vals in dG_ads.items()
    },
    'BEP_corrected': {
        mat: {ads: float(v) for ads,v in vals.items()}
        for mat,vals in dG_corr.items()
    },
    'limiting_potentials': {
        mat: float(v) if not np.isnan(v) else None
        for mat,v in U_L.items()
    },
    'BEP_corrections': BEP,
    'ZPE_TS_corrections': ZPE_TS,
    'DZPE': DZPE,
}
with open('chgnet_analysis_results.json','w') as f:
    json.dump(output, f, indent=2)
print('Saved: chgnet_analysis_results.json')

# ── Format 2: merge_results.py schema ───────────────────────────────────
# Schema: {'model':..., 'systems': {sys: {ads_energies, U_L, pds, ...}}}
report = {'model': 'CHGNet', 'systems': {}}
for mat in dG_corr:
    ae  = dG_corr[mat]
    ul  = U_L.get(mat, np.nan)
    # PDS: step with largest dG at U=0
    if not any(np.isnan(v) for v in ae.values()):
        G_profile = free_energy_profile(ae, U_V=0.0)
        step_names = ['*COOH','*CO','*CHO','*CH2O','*CH3O','CH3OH']
        step_dg = np.diff(G_profile)
        pds_idx = int(np.argmax(step_dg))
        pds_lbl = step_names[pds_idx] if pds_idx < len(step_names) else '?'
    else:
        step_dg = [np.nan]*6; pds_lbl = 'N/A'
    report['systems'][mat] = {
        'ads_energies': {k: round(float(v),4) for k,v in ae.items()},
        'U_L':          round(float(ul),4) if not np.isnan(ul) else None,
        'pds':          pds_lbl,
        'overpotential':round(abs(float(ul))-0.016,4) if not np.isnan(ul) else None,
        'Ea_PDS':       None,   # Marcus barriers not computed in CHGNet notebook
        'BEP_applied':  BEP.get(mat),
        'step_dG':      [round(float(x),4) for x in step_dg],
    }
with open('report_CHGNet.json','w') as f:
    json.dump(report, f, indent=2)
print('Saved: report_CHGNet.json  (merge_results.py compatible)')

# ── Summary ──────────────────────────────────────────────────────────────
print()
print('Limiting potentials (BEP-corrected):')
for mat,ul in U_L.items():
    print(f'  {mat}: U_L = {ul:.3f} V vs RHE')
print()
print('Not yet available: WO3 BEP (requires VASP Gr/WO3 bilayer)')
if BEP.get('WO3') is None:
    print('  - WO3 BEP correction: run VASP on POSCAR_Gr_WO3_bilayer.vasp')
print('  - Torch irradiance: measure with power meter')
print('  - Raw potentiostat CSV: export from Versastat 3')

# ── Format 3: long-form CSV for MACE notebook §18 parity comparison ──────
# Schema: System, intermediate, dEads_eV  — one row per (system, adsorbate).
# Both raw (dG_ads) and BEP-corrected (dG_corr) values are written so the
# parity plot can choose either depending on what was used in the MACE run.
import pandas as pd

rows_raw = []
rows_bep = []
for mat, vals in dG_ads.items():
    for ads, dE in vals.items():
        if dE is None or (isinstance(dE, float) and np.isnan(dE)):
            continue
        rows_raw.append({'System': mat, 'intermediate': ads,
                          'dEads_eV': float(dE)})
for mat, vals in dG_corr.items():
    for ads, dE in vals.items():
        if dE is None or (isinstance(dE, float) and np.isnan(dE)):
            continue
        rows_bep.append({'System': mat, 'intermediate': ads,
                          'dEads_eV': float(dE)})

df_raw_csv = pd.DataFrame(rows_raw)
df_bep_csv = pd.DataFrame(rows_bep)

# The MACE notebook loads from ./MACE_results/adsorption_CHGNet.csv
# We write to the local directory (next to the report JSONs) and also
# place a copy in ./MACE_results if that folder exists.
from pathlib import Path

raw_path = Path('adsorption_CHGNet_raw.csv')
bep_path = Path('adsorption_CHGNet.csv')          # default consumed by MACE nb
df_raw_csv.to_csv(raw_path, index=False)
df_bep_csv.to_csv(bep_path, index=False)
print(f'Saved: {raw_path}  ({len(df_raw_csv)} rows, raw CHGNet dE)')
print(f'Saved: {bep_path}  ({len(df_bep_csv)} rows, CHGNet+BEP)')

mace_dir = Path('MACE_results')
if mace_dir.is_dir():
    df_bep_csv.to_csv(mace_dir / 'adsorption_CHGNet.csv', index=False)
    df_raw_csv.to_csv(mace_dir / 'adsorption_CHGNet_raw.csv', index=False)
    print(f'Also copied to {mace_dir}/')


## 12. Copy-Paste Block for Analysis Notebook

In [ ]:
# Section 12 — Generate paste block AND write structured output for analysis notebook
# The analysis notebook can now load chgnet_analysis_results.json directly
# (see updated Section 5 in analysis notebook).
# This cell also prints the legacy paste block for manual override if needed.

print('# ── Auto-generated paste block (copy to analysis notebook §5 if needed) ──')
print()
print('oxide_dG_CO = {')
for mat, vals in dG_corr.items():
    co = vals.get('CO', np.nan)
    rgo_key = f'{mat}/rGO'
    bep_note = f"CHGNet+BEP+{BEP.get(mat,0):.3f}" if BEP.get(mat) else 'CHGNet (no BEP)'
    print(f'    "{rgo_key}": {co:.4f},  # {bep_note}')
print('}')
print()
print('oxide_UL = {')
for mat,ul in U_L.items():
    rgo_key = f'{mat}/rGO'
    print(f'    "{rgo_key}": {ul:.4f},  # CHGNet+BEP+CHE')
print('}')
print()
print('dG_intermediates = {}')
for mat, vals in dG_corr.items():
    rgo_key = f'{mat}/rGO'
    print(f'dG_intermediates["{rgo_key}"] = {{')
    for ads in ['*COOH','*CO','*CHO','*CH2O','*CH3O']:
        key = ads.replace('*','')
        v = vals.get(key, np.nan)
        bep = BEP.get(mat,0) or 0
        print(f'    "{ads}": {v:.4f},  # CHGNet+BEP')
    print(f'    "BEP_applied": {BEP.get(mat)},  # eV')
    print('}')
    print()
print('# ── NOTE: prefer loading from chgnet_analysis_results.json in analysis nb ──')
